In [87]:
# ============================================================
# 📦 IMPORTS
# Chargement des bibliothèques
# ============================================================

import pandas as pd
import numpy as np
import nltk
from nltk.stem import SnowballStemmer
stem_en = SnowballStemmer("english")
# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.metrics import confusion_matrix

import spacy
# Import des règles anglaises
nlp = spacy.load('en_core_web_sm')

In [38]:
# ============================================================
# 📂 CHARGEMENT DES DONNÉES
# Lecture et premier aperçu du dataset
# ============================================================

def read_csv(fichier):
    df = pd.read_csv(fichier)
    return df

df = read_csv("train.csv")

In [39]:
# ============================================================
# 📌 Suppression des tweets "neutral"
# Je supprime les tweets neutral et je garde positive et negative
# ============================================================

# Je crée une condition pour garder que les lignes sans neutral
condition = df['sentiment'] != 'neutral'
df_sans_neutral = df[condition]


# ============================================================
# 📌 Pourcentage des tweets positifs et negatifs
# 
# ============================================================

tweet_positif = df_sans_neutral['sentiment'] == 'positive'
tweet_negatif = df_sans_neutral['sentiment'] == 'negative'

pourcentage_positif = round((len(df_sans_neutral[tweet_positif]) / len(df_sans_neutral['sentiment']) * 100),2)
print(f"Le pourcentage de tweet positif est de {pourcentage_positif}%")

pourcentage_negatif = round((len(df_sans_neutral[tweet_negatif]) / len(df_sans_neutral['sentiment']) * 100),2)
print(f"Le pourcentage de tweet négatif est de {pourcentage_negatif}%")

Le pourcentage de tweet positif est de 52.45%
Le pourcentage de tweet négatif est de 47.55%


In [ ]:
# ============================================================
# 📌 Création de la fonction clean qui retourne un str de tokens
#  après application d'un stemmer ou lemmatizer, séparé par des espaces
# ============================================================

def clean(texte, methode='lemma'):

        if methode == 'lemma':
            sent_tokens = nlp(texte)
            tokens = []
            for token in sent_tokens:
                if not token.is_stop and not token.is_punct:
                    tokens.append(token.lemma_)
            return " ".join(tokens)

        elif methode == 'stemmer':
            sent_tokens = nlp(texte)
            tokens = []
            for token in sent_tokens:
                if not token.is_stop and not token.is_punct:
                    tokens.append(stem_en.stem(token.text))
            return " ".join(tokens)     

In [41]:
# ============================================================
# 📌 Je teste ma fonciton via un exemple
#  
# ============================================================

clean("You are better when I am well.", methode='stemmer')

'better'

In [42]:
# ============================================================
# 📌 J'ajoute une nouvelle colonne clean a mon dataframe
#   ou j'applique ma fonction
# ============================================================

df_sans_neutral['clean'] = df_sans_neutral['text'].apply(clean)

In [43]:
# ============================================================
# 📌 J'affiche mon df après avoir créé ma nouvelle colonne
#
# ============================================================

df_sans_neutral.head()

,textID,text,selected_text,sentiment,clean
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,Sooo SAD miss San Diego
2,088c60f138,my boss is bullying me...,bullying me,negative,boss bully
3,9642c003ef,what interview! leave me alone,leave me alone,negative,interview leave
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,son couldn`t release buy
6,6e0c6d75b1,2am feedings for the baby are fun when he is a...,fun,positive,2 feeding baby fun smile coo


In [44]:
# ============================================================
# 📌 Je définis X et y
#  J'applique le train_test_split sur mes features et ma cible
# ============================================================

X = df_sans_neutral['clean']
y = df_sans_neutral['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=32)

In [45]:
# ============================================================
# 📌 Je crée un modèle CountVectorizer avec la methode CountVectorizer
#  J'entraine mon modèle sur X_train
# ============================================================

from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
vectorizer.fit(X_train)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [46]:
# ============================================================
# 📌 Je crée un modèle TfidfVectorizer avec la methode TfidfVectorizer
#  J'entraine mon modèle sur X_train
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
tfidf.fit(X_train)



,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1

In [47]:
# ============================================================
# 📌 Je crée une matrice de features et ma matrice X_test_CV
#  J'entraine mon X_train et je crée mes matmrices
# ============================================================

X_train_CV = vectorizer.fit_transform(X_train)
X_test_CV = vectorizer.transform(X_test)

print(X_test_CV)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 24654 stored elements and shape (4091, 13656)>
  Coords	Values
  (0, 10822)	1
  (0, 11458)	1
  (1, 1656)	1
  (1, 3592)	1
  (1, 3787)	1
  (1, 5213)	1
  (1, 6417)	1
  (1, 8536)	1
  (1, 8722)	1
  (1, 9213)	1
  (1, 10346)	1
  (1, 11083)	1
  (1, 12099)	1
  (1, 13061)	1
  (2, 811)	1
  (2, 1724)	1
  (2, 2839)	1
  (2, 2887)	1
  (2, 10711)	1
  (2, 12698)	1
  (3, 2670)	1
  (3, 3359)	1
  (3, 4517)	1
  (3, 5213)	1
  (3, 7382)	1
  :	:
  (4084, 8008)	1
  (4084, 8090)	1
  (4084, 9117)	1
  (4084, 11296)	1
  (4084, 11724)	1
  (4084, 12677)	1
  (4084, 13104)	1
  (4085, 3302)	1
  (4085, 8501)	1
  (4085, 10589)	1
  (4085, 11724)	1
  (4085, 12395)	1
  (4086, 2318)	1
  (4087, 5912)	1
  (4087, 11198)	1
  (4088, 12947)	1
  (4089, 6042)	1
  (4090, 1610)	1
  (4090, 1624)	1
  (4090, 1922)	1
  (4090, 7068)	1
  (4090, 8186)	1
  (4090, 8408)	1
  (4090, 11149)	1
  (4090, 11899)	1


In [48]:
# ============================================================
# 📌 Je crée une matrice de features et ma matrice X_test_CV
#  J'entraine mon X_train et je crée mes matmrices
# ============================================================

X_train_CV2 = tfidf.fit_transform(X_train)
X_test_CV2 = tfidf.transform(X_test)

print(X_test_CV2)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 24654 stored elements and shape (4091, 13656)>
  Coords	Values
  (0, 10822)	0.8031550479175126
  (0, 11458)	0.5957700638708009
  (1, 1656)	0.34372516900615646
  (1, 3592)	0.3019922350314974
  (1, 3787)	0.24987419150104193
  (1, 5213)	0.20690240261752016
  (1, 6417)	0.22761992022497962
  (1, 8536)	0.23782207524439364
  (1, 8722)	0.3200731281879781
  (1, 9213)	0.3504897334783543
  (1, 10346)	0.2639245307171643
  (1, 11083)	0.3624798827904271
  (1, 12099)	0.27079966462128763
  (1, 13061)	0.2781504493355845
  (2, 811)	0.3738525809478721
  (2, 1724)	0.38460682688982184
  (2, 2839)	0.24602359499938764
  (2, 2887)	0.3896117930954129
  (2, 10711)	0.48923411499008207
  (2, 12698)	0.510526060591213
  (3, 2670)	0.572704389800649
  (3, 3359)	0.24373288763428222
  (3, 4517)	0.48195785561845444
  (3, 5213)	0.273383954467751
  (3, 7382)	0.2767112143892988
  :	:
  (4084, 8008)	0.26234198507276957
  (4084, 8090)	0.23253939672682952
  (4084, 

In [ ]:
# ============================================================
# 📌 J'entraine avec une LogisticRegression
#  avec le CountVectorizer
# ============================================================

def train_model(X_train, y_train, modele='logistic'):
    """
    Utilisation : model = train_model(X_train, y_train)
    Modeles disponibles : 'logistic', 'decision_tree', 'random_forest', 'linear', 'KNN'
    """
    if modele == 'logistic':
        model = LogisticRegression()
    elif modele == 'decision_tree':
        model = DecisionTreeClassifier()
    elif modele == 'random_forest':
        model = RandomForestClassifier()
    elif modele == 'linear':
        model = LinearRegression()
    elif modele == 'KNN':
        model = KNeighborsClassifier()
    model.fit(X_train, y_train)
    return model

model = train_model(X_train_CV, y_train, modele='logistic')

In [50]:
# ============================================================
# 📌 Le modèle est entrainé
# Maintenant je calcule les scores
# ============================================================

print(f"Le modele avec le CountVectorizer obtient un accuracy de {model.score(X_train_CV, y_train)} sur le train et {model.score(X_test_CV, y_test)} sur le test")

Le modele avec le CountVectorizer obtient un accuracy de 0.9551825293350718 sur le train et 0.8692251283304816 sur le test


In [ ]:
# ============================================================
# 📌 J'entraine avec une LogisticRegression
#  avec le TfidfVectorizer
# ============================================================

def train_model(X_train, y_train, modele='logistic'):
    """
    Utilisation : model = train_model(X_train, y_train)
    Modeles disponibles : 'logistic', 'decision_tree', 'random_forest', 'linear', 'KNN'
    """
    if modele == 'logistic':
        model = LogisticRegression()
    elif modele == 'decision_tree':
        model = DecisionTreeClassifier()
    elif modele == 'random_forest':
        model = RandomForestClassifier()
    elif modele == 'linear':
        model = LinearRegression()
    elif modele == 'KNN':
        model = KNeighborsClassifier()
    model.fit(X_train, y_train)
    return model

model = train_model(X_train_CV2, y_train, modele='logistic')

In [52]:
# ============================================================
# 📌 Le modèle est entrainé
# Maintenant je calcule les scores
# ============================================================

print(f"Le modele avec le TfidfVectorizer obtient un accuracy de {model.score(X_train_CV2, y_train)} sur le train et {model.score(X_test_CV2, y_test)} sur le test")

Le modele avec le TfidfVectorizer obtient un accuracy de 0.9278846153846154 sur le train et 0.8704473233928135 sur le test


In [ ]:

# Les deux modèles ont des écarts train/test similaires,  donc pas d'overfitting particulier dans les deux cas. 
# Le CountVectorizer est globalement meilleur car il pondère mieux l'importance des mots.

In [ ]:
# ====================================================== AUTRES MODELES DE CLASSIFICATION + BONUS =========================================================

In [55]:
# Je reprends ma fonction et je garde les points d'exclamation pour voir la difference

def clean2(texte, methode='lemma'):

        if methode == 'lemma':
            sent_tokens = nlp(texte)
            tokens = []
            for token in sent_tokens:
                if not token.is_stop and (not token.is_punct or token.text == '!'):
                    tokens.append(token.lemma_)
            return " ".join(tokens)

        elif methode == 'stemmer':
            sent_tokens = nlp(texte)
            tokens = []
            for token in sent_tokens:
                if not token.is_stop and (not token.is_punct or token.text == '!'):
                    tokens.append(stem_en.stem(token.text))
            return " ".join(tokens)     

In [56]:
df_excla = df_sans_neutral.copy()

In [58]:
# ============================================================
# 📌 J'ajoute une nouvelle colonne clean a mon dataframe
#   ou j'applique ma fonction qui garde les "!"
# ============================================================

df_excla['clean'] = df_excla['text'].apply(clean2)

In [59]:
df_excla.head()

,textID,text,selected_text,sentiment,clean
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,Sooo SAD miss San Diego ! ! !
2,088c60f138,my boss is bullying me...,bullying me,negative,boss bully
3,9642c003ef,what interview! leave me alone,leave me alone,negative,interview ! leave
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,son couldn`t release buy
6,6e0c6d75b1,2am feedings for the baby are fun when he is a...,fun,positive,2 feeding baby fun smile coo


In [60]:
# ============================================================
# 📌 Je définis X et y
#  J'applique le train_test_split sur mes features et ma cible
# ============================================================

X = df_excla['clean']
y = df_excla['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=32)

In [63]:
# ============================================================
# 📌 Je crée un modèle CountVectorizer avec la methode CountVectorizer
#  J'entraine mon modèle sur X_train
# ============================================================

from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
vectorizer.fit(X_train)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [65]:
# ============================================================
# 📌 Je crée une matrice de features et ma matrice X_test_CV
#  J'entraine mon X_train et je crée mes matrices
# ============================================================

X_train_CV = vectorizer.fit_transform(X_train)
X_test_CV = vectorizer.transform(X_test)


In [ ]:
model2 = train_model(X_train_CV, y_train, modele='logistic')

In [76]:
# ============================================================
# 📌 Le modèle est entrainé
# Maintenant je calcule les scores
# ============================================================

print(f"Le modele avec le CountVectorizer obtient un accuracy de {round(model2.score(X_train_CV, y_train),3)} sur le train et {round(model2.score(X_test_CV, y_test),3)} sur le test")

Le modele avec le CountVectorizer obtient un accuracy de 0.955 sur le train et 0.869 sur le test


In [ ]:
# Les résultats sont identiques, je serais curieux de savoir si ca a marché chez quelqu'un d'autre ou si j'ai mal fait quelque chose.